# Sentence Transformers: find an answer even when the wording changes

A customer asks “How do I get my money back?” but the help article is titled “Refund status.” Keyword overlap can miss a relevant answer; a sentence encoder offers another representation. We will compare lexical and semantic retrieval, inspect failure cases, evaluate held-out questions and decide when a search system should abstain.

The dataset is a fictional help center authored in this notebook. The encoder is the real pretrained `sentence-transformers/all-MiniLM-L6-v2`, pinned to an immutable Hub revision. Unlike the synthetic model in Transformers.ipynb, this encoder already has learned text representations. It still does not verify facts or guarantee that a retrieved article answers a question.

All teaching material, examples and the complete project live here. **The first setup needs internet to download the public model (roughly 90 MB); later runs reuse the local cache.** No API key, paid endpoint, server or GPU is required. Read a section, predict the result, run it, and explain what its evidence supports.

## 1. Separate the encoder, search algorithm and decision rule

An encoder converts text into vectors; a similarity calculation ranks vectors; a decision rule determines whether to show a result. These are different components and need different checks. Sentence Transformers supplies sentence encoders and related training/evaluation tools. FAISS or Qdrant can search the resulting vectors at larger scale.

Use `sentence-transformers==5.1.2` with the repository's PyTorch, Transformers, numpy, pandas, scikit-learn and matplotlib. These pins support its Python 3.9 environment. The import uses an underscore: `sentence_transformers`. We select the PyTorch backend and CPU explicitly. Restart the kernel if you previously imported Transformers with different backend settings.

Reports go to `outputs/sentence_transformers/` and are replaced on rerun. Cached model weights go under `data/cache/` and are reused, not committed to the repository.

In [ ]:
import os
os.environ['USE_TF']='0'
os.environ['USE_TORCH']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HUB_DISABLE_XET']='1'
from pathlib import Path
import json, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import sentence_transformers
from sentence_transformers import SentenceTransformer, util
from huggingface_hub import snapshot_download
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from IPython.display import display

OUT=Path('outputs/sentence_transformers');OUT.mkdir(parents=True,exist_ok=True)
torch.set_num_threads(2)
print({'python':platform.python_version(),'sentence_transformers':sentence_transformers.__version__,'torch':torch.__version__})

## 2. Load a specific model instead of a moving name

The model card describes this checkpoint as an English sentence/short-paragraph encoder with 384-dimensional output, an Apache-2.0 license and a default 256-wordpiece limit. We inspect the runtime configuration instead of assuming every sentence encoder shares those properties. [Model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).

The code checks a revision marker and required files before downloading. Set `ALLOW_DOWNLOAD=False` when you want a strictly local run. If the cache is absent, it raises a clear error rather than substituting random or lexical vectors. Preserve this cache when moving the notebook to an offline machine.

`trust_remote_code=False` and the explicit module files use standard installed implementations. Loading the matching pooling configuration matters: a generic Transformer token output is not automatically a useful sentence embedding.

In [ ]:
MODEL_ID='sentence-transformers/all-MiniLM-L6-v2'
REVISION='1110a243fdf4706b3f48f1d95db1a4f5529b4d41'
MODEL_DIR=Path('data/cache/all-MiniLM-L6-v2')
ALLOW_DOWNLOAD=True
required=['config.json','config_sentence_transformers.json','sentence_bert_config.json','modules.json',
          '1_Pooling/config.json','model.safetensors','tokenizer.json','tokenizer_config.json',
          'special_tokens_map.json','vocab.txt']
marker=MODEL_DIR/'notebook_revision.txt'
ready=marker.exists() and marker.read_text().strip()==REVISION and all((MODEL_DIR/f).is_file() for f in required)
if not ready:
    if not ALLOW_DOWNLOAD:
        raise FileNotFoundError(f'Pinned model cache missing at {MODEL_DIR}. Run once online with ALLOW_DOWNLOAD=True.')
    try:
        snapshot_download(MODEL_ID,revision=REVISION,local_dir=str(MODEL_DIR),
                          allow_patterns=required+['README.md'],max_workers=2)
    except Exception as exc:
        raise RuntimeError(f'Could not cache {MODEL_ID} at revision {REVISION}. Check network access, then rerun this cell.') from exc
    assert all((MODEL_DIR/f).is_file() for f in required)
    marker.write_text(REVISION+'\n')
model=SentenceTransformer(str(MODEL_DIR),device='cpu',local_files_only=True,trust_remote_code=False)
print(model)
print({'dimension':model.get_sentence_embedding_dimension(),'max_seq_length':model.max_seq_length,'prompts':model.prompts})
assert model.get_sentence_embedding_dimension()==384

## 3. Define what counts as a relevant answer before searching

One row is a short help article with a stable document ID, title, body and topic. There are 16 articles. Twelve authored queries are split into development and test sets: four answerable and two unanswerable queries in each split.

Some questions have two acceptable documents. These are explicit teaching judgments, not exhaustive expert relevance labels. No query labels are used to fit the pretrained encoder or lexical vocabulary. Development queries can guide the abstention threshold; test queries are held aside until that threshold is fixed.

All articles are assumed to be eligible for the same reader. In a multi-tenant system, apply ownership/publication filters as in Qdrant.ipynb; semantic similarity cannot enforce access rules.

In [ ]:
articles=[
 ('a01','Reset a password','Use the recovery email to choose a new password when you cannot sign in.','account'),
 ('a02','Two factor recovery','Use a saved backup code if your authentication phone is lost.','account'),
 ('a03','Change account email','Verify the new inbox before replacing the email address on your profile.','account'),
 ('a04','Close your account','Request permanent account deletion after downloading any records you need.','account'),
 ('b01','Refund status','A returned payment goes back to the original card; check its progress in payment history.','billing'),
 ('b02','Duplicate charge','Report being charged twice for the same purchase so billing can investigate.','billing'),
 ('b03','Invoice download','Download a purchase receipt or invoice from your billing history.','billing'),
 ('b04','Replace payment card','Update expired credit card details before the next renewal.','billing'),
 ('d01','Shipment tracking','Follow the courier tracking link to see where your parcel is.','delivery'),
 ('d02','Broken delivery','Send photos of damaged items and packaging to request assistance.','delivery'),
 ('d03','Change shipping address','Correct the destination before the parcel leaves the warehouse.','delivery'),
 ('d04','Missing parcel','Contact support when the courier marks an order delivered but it never reached you.','delivery'),
 ('s01','Stop renewal','Cancel your subscription to prevent future recurring charges.','subscription'),
 ('s02','Pause membership','Temporarily suspend your plan and resume it next month.','subscription'),
 ('s03','Upgrade membership','Move to a higher plan for more storage and additional team seats.','subscription'),
 ('s04','Resume membership','Restart a paused subscription from the membership settings page.','subscription')
]
docs=pd.DataFrame(articles,columns=['doc_id','title','text','topic'])
queries=pd.DataFrame([
 ('I forgot my login secret. How can I get in?',{'a01'},'development'),
 ('My bank statement shows the same purchase two times.',{'b02'},'development'),
 ('The box arrived with shattered contents.',{'d02'},'development'),
 ('I need a break from paying for my plan.',{'s01','s02'},'development'),
 ('Will it rain in London tomorrow?',set(),'development'),
 ('How do I bake sourdough bread?',set(),'development'),
 ('Can I obtain proof that I paid?',{'b03'},'test'),
 ('My parcel went to the wrong house. Who can help?',{'d03','d04'},'test'),
 ('I lost the phone used for login verification.',{'a02'},'test'),
 ('I want to restart the membership I put on hold.',{'s04'},'test'),
 ('Who won the last football world cup?',set(),'test'),
 ('What medicine cures a headache?',set(),'test')
],columns=['query','relevant','split'])
assert docs.doc_id.is_unique and len(docs)==16
assert all(r.issubset(set(docs.doc_id)) for r in queries.relevant)
display(docs)
display(queries.assign(relevant=queries.relevant.map(sorted)))

## 4. Check input lengths before trusting short-text embeddings

The encoder truncates at its configured token limit. A document can look intact in a DataFrame while its decisive final sentence never reaches the network. Count wordpieces, including special tokens, before choosing a chunking strategy.

Our articles fit within the limit. For a long demonstration paragraph, construct overlapping token windows, keeping space for special tokens. This mechanical tokenizer-based splitter is illustrative; real chunking should preserve section boundaries and source IDs. Overlap can duplicate evidence and crowd the top results.

In [ ]:
texts=(docs.title+'. '+docs.text).tolist()
lengths=[len(model.tokenizer(t,add_special_tokens=True,truncation=False)['input_ids']) for t in texts]
assert max(lengths)<=model.max_seq_length
print('Maximum article tokens:',max(lengths))
long_text=' '.join(texts*4)
raw_tokens=model.tokenizer(long_text,add_special_tokens=False,truncation=False)['input_ids']
window=48;overlap=8
assert window+model.tokenizer.num_special_tokens_to_add(False)<=model.max_seq_length
chunks=[]
for start in range(0,len(raw_tokens),window-overlap):
    end=min(start+window,len(raw_tokens))
    chunks.append({'source_id':'long-demo','start_token':start,'end_token':end,
                   'text':model.tokenizer.decode(raw_tokens[start:end],skip_special_tokens=True)})
    if end==len(raw_tokens): break
assert chunks[-1]['end_token']==len(raw_tokens)
display(pd.DataFrame(chunks).head(3))

## 5. Inspect a lexical reference before attributing gains to semantics

Fit TF-IDF on the corpus only. It provides a useful inexpensive baseline: shared words and word pairs can be very strong evidence in a help center. A pretrained model is not automatically preferable for every query or domain.

Predict whether a query with no corpus vocabulary gets meaningful lexical scores. The baseline below reports zero similarities for that case. We keep the result visible instead of silently claiming that its top arbitrary tie is an answer.

In [ ]:
lexical=TfidfVectorizer(ngram_range=(1,2))
corpus_sparse=lexical.fit_transform(texts)
lexical_scores=(lexical.transform(queries['query']) @ corpus_sparse.T).toarray()
example_index=1
order=np.argsort(-lexical_scores[example_index],kind='stable')[:3]
display(docs.iloc[order][['doc_id','title']].assign(score=lexical_scores[example_index,order]))

## 6. Encode documents and queries with the same checkpoint

Use batches to amortize inference overhead and normalize embeddings to unit length. `encode_document` and `encode_query` communicate their roles; models with role-specific prompts may use different instructions. This checkpoint has no task prompts, so those methods use the same basic encoder. Do not invent an `E5`-style prompt for an unrelated checkpoint.

Each output row must still correspond to the same document ID. Check shape, finiteness and norms before computing similarities. Changing the checkpoint, pooling, prompts or normalization means changing the representation version, even if vector dimensions happen to match.

In [ ]:
embeddings=model.encode_document(texts,batch_size=8,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False)
query_embeddings=model.encode_query(queries['query'].tolist(),batch_size=8,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False)
assert embeddings.shape==(len(docs),384)
assert query_embeddings.shape==(len(queries),384)
assert np.isfinite(embeddings).all() and np.isfinite(query_embeddings).all()
assert np.allclose(np.linalg.norm(embeddings,axis=1),1,atol=1e-5)
semantic_scores=query_embeddings @ embeddings.T
assert np.allclose(semantic_scores,util.cos_sim(query_embeddings,embeddings).numpy(),atol=1e-5)
print('Score matrix:',semantic_scores.shape)

## 7. Return evidence with the ranked IDs

For normalized vectors, the dot product equals cosine similarity. A larger score means closer vectors under this model, not a probability that the text is correct or relevant. Display the article body so a reader can inspect the evidence.

On a small corpus, exact matrix multiplication is sufficient. `util.semantic_search` can process searches in chunks; chunking reduces peak intermediate memory, but it does not turn exact search into an approximate index. Compare its top result with the direct calculation.

In [ ]:
def ranked_rows(scores,k=3):
    if k<1:raise ValueError('k must be positive')
    order=np.argsort(-scores,kind='stable')[:min(k,len(docs))]
    return docs.iloc[order][['doc_id','title','text']].assign(score=scores[order]).reset_index(drop=True)

display(ranked_rows(semantic_scores[0]))
hits=util.semantic_search(query_embeddings,embeddings,top_k=3)
assert all(h[0]['corpus_id']==int(np.argmax(row)) for h,row in zip(hits,semantic_scores))
# Inspect disagreement instead of assuming the neural ranking is always better.
comparison=pd.DataFrame({'query':queries['query'],'lexical_top':docs.iloc[lexical_scores.argmax(axis=1)].doc_id.to_numpy(),
                         'semantic_top':docs.iloc[semantic_scores.argmax(axis=1)].doc_id.to_numpy()})
display(comparison)

## 8. Give retrieval metrics explicit denominators

For answerable questions, Recall@3 is the fraction of designated relevant articles in the top three; Hit@3 asks whether any relevant article appears. MRR@3 averages the reciprocal rank of the first relevant article, with zero if absent.

Unanswerable questions have no relevant set, so recall is undefined rather than automatically zero. We evaluate them later through false acceptance. Keep per-query results because a six-query split is too small for stable general conclusions.

The model and representation were fixed before examining either split. Only the abstention threshold is chosen on development data in the next section. This evaluates an existing encoder; it does not fine-tune it.

In [ ]:
metric_rows=[]
for name,scores in [('TF-IDF',lexical_scores),('MiniLM',semantic_scores)]:
    for i,row in queries.iterrows():
        if not row.relevant:continue
        retrieved=ranked_rows(scores[i]).doc_id.tolist()
        ranks=[rank+1 for rank,doc_id in enumerate(retrieved) if doc_id in row.relevant]
        metric_rows.append({'method':name,'split':row['split'],'query':row['query'],
                            'recall_at_3':len(set(retrieved)&row.relevant)/len(row.relevant),
                            'hit_at_3':int(bool(ranks)),'reciprocal_rank_at_3':1/min(ranks) if ranks else 0.0})
metrics=pd.DataFrame(metric_rows)
display(metrics)
display(metrics.groupby(['split','method'])[['recall_at_3','hit_at_3','reciprocal_rank_at_3']].mean())

## 9. Decide when returning nothing is the better behavior

The nearest document always exists in a nonempty corpus, including for a weather question. We choose a cosine threshold using development questions only. A returned top-one document is correct only if its ID is in the query's relevant set.

Our illustrative cost is 3 for an accepted wrong answer and 1 for declining an answerable question; declining an unanswerable question costs zero. An accepted wrong answer counts under the first rule only. These costs are a teaching policy, not an empirically calibrated business value. Among tied development costs, prefer the higher threshold.

With six development questions this threshold is fragile. A real deployment needs representative positive and no-answer questions, subgroup checks and changes monitored over time. Cosine scores are not calibrated probabilities.

In [ ]:
top_positions=semantic_scores.argmax(axis=1)
top_scores=semantic_scores.max(axis=1)
top_ids=docs.iloc[top_positions].doc_id.to_numpy()
answerable=queries.relevant.map(bool).to_numpy()
correct_top=np.array([doc_id in relevant for doc_id,relevant in zip(top_ids,queries.relevant)])
dev=(queries['split']=='development').to_numpy()
threshold_rows=[]
for threshold in np.r_[np.linspace(-1,1,81),1.01]:
    accepted=top_scores>=threshold
    cost=3*np.sum(accepted[dev]&~correct_top[dev])+np.sum(~accepted[dev]&answerable[dev])
    threshold_rows.append({'threshold':float(threshold),'development_cost':int(cost)})
threshold_table=pd.DataFrame(threshold_rows)
chosen=threshold_table.sort_values(['development_cost','threshold'],ascending=[True,False]).iloc[0]
threshold=float(chosen.threshold)
print('Chosen from development only:',chosen.to_dict())
accepted=top_scores>=threshold
decisions=queries[['query','split']].copy()
decisions['top_doc']=top_ids;decisions['top_score']=top_scores
decisions['answerable']=answerable;decisions['correct_top']=correct_top;decisions['accepted']=accepted
display(decisions)
test_mask=(queries['split']=='test').to_numpy()
print({'test_accepted':int(accepted[test_mask].sum()),
       'test_wrong_accepts':int((accepted[test_mask]&~correct_top[test_mask]).sum()),
       'test_answerable_declines':int((~accepted[test_mask]&answerable[test_mask]).sum())})
fig,ax=plt.subplots(figsize=(8,4))
for label,mask in [('answerable',answerable),('no answer',~answerable)]:
    ax.scatter(np.flatnonzero(mask),top_scores[mask],label=label)
ax.axhline(threshold,color='black',linestyle='--',label='Development threshold')
ax.set(xlabel='Query row',ylabel='Top cosine score',title='High similarity is not a correctness guarantee')
ax.legend();plt.tight_layout();plt.savefig(OUT/'query_scores.png');plt.show()

## 10. Inspect similarity without confusing it with logical truth

Embeddings often place a sentence and its negation near one another because they share topic and wording. Compare a paraphrase, a contradiction and an unrelated sentence. Interpret the numeric pattern you actually see; do not assume a particular ordering is guaranteed.

For duplicate detection, semantic proximity is a candidate-generation step, not proof that two records are interchangeable. For an answer assistant, retrieve source text and validate its relevance; embedding similarity does not establish entailment or factual correctness.

In [ ]:
statements=['The subscription can be cancelled at any time.',
            'You may stop your membership whenever you want.',
            'The subscription cannot be cancelled at any time.',
            'The parcel was damaged during shipping.']
statement_vectors=model.encode(statements,normalize_embeddings=True,show_progress_bar=False)
display(pd.DataFrame(statement_vectors @ statement_vectors.T,index=['original','paraphrase','negation','unrelated'],
                     columns=['original','paraphrase','negation','unrelated']).round(3))

## 11. Use clusters to inspect a corpus, not invent ground truth

Four topics were authored into our help center, so we explore four KMeans clusters. The clustering algorithm does not see those topic labels. Compare the resulting assignments with titles and ask where boundaries are ambiguous, such as recurring charges versus subscriptions.

This is descriptive corpus exploration on unit vectors. It is not evidence that all real help centers have four categories or that the clusters represent causal groups. Cluster IDs have no inherent order or stable business meaning across refits.

In [ ]:
cluster_ids=KMeans(n_clusters=4,random_state=42,n_init=10).fit_predict(embeddings)
cluster_report=docs[['doc_id','title','topic']].assign(cluster=cluster_ids)
display(cluster_report.sort_values(['cluster','doc_id']))
display(pd.crosstab(cluster_report.cluster,cluster_report.topic))

## 12. Keep vectors, IDs and encoder provenance together

Cache corpus embeddings so later searches do not re-encode every article. Save the ordered IDs, text digest, encoder revision and threshold selection policy alongside the matrix. Reload both the matrix and the pinned local encoder and verify fixed-query scores.

A NumPy matrix by itself cannot tell you which row belongs to which article. Reordering text without reordering vectors silently corrupts retrieval. Matching dimensions alone also cannot detect a different checkpoint or preprocessing policy.

At larger scale, place these same normalized vectors in FAISS or Qdrant with stable IDs; benchmark index recall separately from relevance. A bi-encoder embeds queries/documents independently. A cross-encoder can rerank a smaller candidate set jointly at higher per-pair cost, but cannot recover a relevant article that was never retrieved.

In [ ]:
corpus_digest=hashlib.sha256(json.dumps(docs.to_dict(orient='records'),sort_keys=True).encode()).hexdigest()
np.savez_compressed(OUT/'corpus_vectors.npz',embeddings=embeddings,doc_ids=docs.doc_id.to_numpy(dtype=str))
docs.to_json(OUT/'articles.json',orient='records',indent=2)
manifest={'model_id':MODEL_ID,'revision':REVISION,'normalize_embeddings':True,'dimension':384,
          'max_seq_length':model.max_seq_length,'corpus_sha256':corpus_digest,'threshold':threshold,
          'threshold_source':'six authored development queries; cost=3*wrong_accept+answerable_decline',
          'sentence_transformers':sentence_transformers.__version__}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
with np.load(OUT/'corpus_vectors.npz',allow_pickle=False) as saved:
    loaded_vectors=saved['embeddings'];loaded_ids=saved['doc_ids']
assert loaded_ids.tolist()==docs.doc_id.tolist()
reloaded_model=SentenceTransformer(str(MODEL_DIR),device='cpu',local_files_only=True,trust_remote_code=False)
check_query=reloaded_model.encode_query([queries.iloc[0]['query']],normalize_embeddings=True,show_progress_bar=False)
assert np.allclose(check_query @ loaded_vectors.T,semantic_scores[:1],atol=1e-5)
metrics.to_csv(OUT/'relevance_metrics.csv',index=False)
decisions.to_csv(OUT/'query_decisions.csv',index=False)
cluster_report.to_csv(OUT/'cluster_inspection.csv',index=False)
print('Embedding IDs, cached encoder and scores survived reload.')

## 13. Understand training before adapting an encoder

This notebook performs inference and evaluation, not fine-tuning. Domain adaptation needs training pairs/triplets and held-out relevance judgments. Contrastive objectives encourage positive pairs to be close relative to negatives. Random negatives can be too easy; hard negatives can teach useful distinctions but may contain unlabeled true positives.

Multiple-negatives ranking losses commonly treat other items in a batch as negatives, so duplicate or related positives can violate that assumption. Cross-encoders score pairs jointly; they are useful for reranking but have a different cost model from precomputed corpus vectors. Sparse encoders and hybrid retrieval add another representation choice.

Separate all model-fitting data from development threshold tuning and final evaluation. Split related queries/documents by source or entity when near duplicates could leak. Record the exact base checkpoint, objective, negative-generation policy and resulting encoder revision. Rebuild the corpus vectors after changing the encoder.

For multilingual or long-document work, select a suitable checkpoint and evaluate it on the actual language and length distributions. This short English demonstration cannot establish those capabilities.

## 14. Explain what the evidence supports

1. Find a query where TF-IDF and MiniLM disagree. Read both articles and explain which judgment is supported; do not automatically favor the neural model.
2. Add a genuinely new no-answer development set. Predict how stronger false-answer costs would affect coverage, then choose a new threshold without consulting a new final test set.
3. Put a decisive answer beyond the token limit and compare full-text truncation with source-preserving chunks. Aggregate chunk hits by source so overlap does not inflate recall.
4. Compare retrieval-only and retrieve-then-rerank designs using the same eligible corpus and untouched questions. Include latency, memory and candidate recall.

| Observation | Investigate |
|---|---|
| First run cannot load the model | Network, pinned revision, cache completeness and `ALLOW_DOWNLOAD` |
| Scores change after moving machines | Library versions, encoder revision, dtype and preprocessing |
| Long documents retrieve poorly | Truncation, chunk boundaries and source aggregation |
| Unanswerable questions get confident-looking scores | Threshold calibration and no-answer coverage |
| New vector index returns wrong text | ID/row alignment and encoder compatibility |

Official references: [semantic search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html), [encoding API](https://www.sbert.net/docs/package_reference/sentence_transformer/SentenceTransformer.html), [retrieve and rerank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html), and [training overview](https://www.sbert.net/docs/sentence_transformer/training_overview.html). The API and checkpoint versions used here are recorded in the manifest.

## Second project: Retrieve the answer buried after a long introduction

Extend the pinned MiniLM project with a deliberately messy handbook: duplicated imports, blank text, obsolete revisions and long repeated boilerplate. Compare whole-document encoding with bounded token chunks using source-level relevance judgments.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through why valid embeddings can hide missing evidence

MiniLM truncates inputs beyond its configured maximum length. An embedding can be finite and normalized while the answer near the end was never read. Whole-document retrieval and chunk retrieval therefore have different information budgets. Deduplicate identical deliveries, select published revisions explicitly, and inspect token lengths before encoding. Do not infer publication status from similarity.

This section reuses the pretrained `model` loaded earlier. The fixture deliberately puts answers late; it is a diagnostic stress test, not an unbiased retrieval benchmark.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_handbook=[
 ('refund','Refunds return to the original payment card within seven business days.'),
 ('login','Recover your account using a backup authentication code when your phone is lost.'),
 ('delivery','Report a damaged parcel with photographs to request a replacement shipment.'),
 ('cancel','Cancel renewal in the subscription settings before the next billing date.')]
p2_boilerplate='This handbook describes general company procedures and responsibilities. '*80
p2_raw=[{'id':key,'revision':2,'status':'published','text':p2_boilerplate+answer} for key,answer in p2_handbook]
p2_raw += [dict(p2_raw[0]),{'id':'refund','revision':1,'status':'retired','text':'Refunds are never available.'},{'id':'empty','revision':2,'status':'published','text':' '}]
p2_docs=pd.DataFrame(p2_raw).drop_duplicates()
p2_docs=p2_docs[(p2_docs.status=='published') & p2_docs.text.str.strip().ne('')].reset_index(drop=True)
assert len(p2_docs)==4 and p2_docs.id.is_unique
p2_chunks=[]
for row in p2_docs.itertuples():
    tokens=model.tokenizer(row.text,add_special_tokens=False,truncation=False)['input_ids']
    for start in range(0,len(tokens),80):
        part=tokens[start:start+96]
        p2_chunks.append({'id':row.id,'text':model.tokenizer.decode(part,skip_special_tokens=True),'start_token':start})
p2_chunks=pd.DataFrame(p2_chunks)
p2_lengths=[len(model.tokenizer(t,add_special_tokens=True)['input_ids']) for t in p2_chunks.text]
assert max(p2_lengths)<=model.max_seq_length
print({'raw_rows':len(p2_raw),'eligible_documents':len(p2_docs),'chunks':len(p2_chunks),'max_model_tokens':model.max_seq_length,'first_document_tokens':len(model.tokenizer(p2_docs.text[0],truncation=False)['input_ids'])})

### Compare at the document level, and account for extra work

The four questions and their relevant source IDs are authored before ranking. For chunk retrieval, take the maximum chunk score within each document, then rank distinct documents. Otherwise a long document could occupy all top positions. Maximum scoring can favor documents with more chunks, so it is a competing approach with a measurable bias rather than a universal fix.

We freeze the 96-token window and 16-token overlap before these diagnostic queries. Tokenization may change when decoded text is encoded again, which is why the code asserts the actual final lengths.

In [ ]:
p2_questions=['When will money return to my payment card?','How can I sign in after losing my phone?','What evidence do I need for a broken package?','Where can I stop my membership renewing?']
p2_relevant=['refund','login','delivery','cancel']
p2_q=model.encode(p2_questions,batch_size=4,normalize_embeddings=True,show_progress_bar=False)
p2_results=[]
for p2_name,p2_frame in [('whole documents',p2_docs),('token chunks',p2_chunks)]:
    p2_start=time.perf_counter()
    p2_emb=model.encode(p2_frame.text.tolist(),batch_size=8,normalize_embeddings=True,show_progress_bar=False)
    p2_elapsed=time.perf_counter()-p2_start
    p2_scores=p2_q@p2_emb.T
    for i,target in enumerate(p2_relevant):
        ranked=pd.DataFrame({'id':p2_frame.id,'score':p2_scores[i]}).groupby('id').score.max().sort_values(ascending=False,kind='stable')
        p2_results.append({'approach':p2_name,'query':p2_questions[i],'top_document':ranked.index[0],'correct_at_1':int(ranked.index[0]==target),'encoding_seconds':p2_elapsed,'vector_bytes':p2_emb.nbytes,'encoded_items':len(p2_emb)})
display(pd.DataFrame(p2_results));display(pd.DataFrame(p2_results).groupby('approach').correct_at_1.mean())

### Diagnose the result before claiming an improvement

Whole-document truncation can make these four documents nearly indistinguishable because their prefixes are identical. Chunking exposes the answer, at the price of more vectors, encoding work and opportunities for a spurious maximum. The displayed encoding time is shared across the four questions for each approach; do not add it four times.

**Try:** put a contradictory retired answer in a high-scoring chunk. **Worked reasoning:** semantic similarity cannot resolve publication authority. Keep revision filtering ahead of ranking, attach source IDs to every result, and assess a larger held-out query set before changing a production retriever. Compare lexical retrieval as a cheap fallback for exact names and codes.